# Pre-entrega 4: Sistema RAG escalable en la nube con Pinecone
Ingesta en un índice Serverless de Pinecone, recuperador híbrido (BM25 + vectorial) y evaluación con Precision@5 y Recall@5.


## 1. Instalación de librerías

In [4]:
!pip install -q pinecone langchain-pinecone langchain-huggingface langchain-text-splitters langchain-community langchain-classic rank_bm25 sentence-transformers tiktoken

## 2. Configuración

In [10]:
import os
from google.colab import userdata

os.environ["PINECONE_API_KEY"] = userdata.get("PINECONE_API_KEY")

INDEX_NAME = "curso-ai-engineering"
NAMESPACE = "fastapi-docs"          # separa este dataset de otros dentro del índice
CLOUD = "aws"
REGION = "us-east-1"                # la región que permite el plan gratuito

EMBEDDINGS_MODEL = "intfloat/multilingual-e5-small"

DATA_DIR = "data"
CHUNK_SIZE = 600      # tokens (entre 500 y 800)
CHUNK_OVERLAP = 60    # tokens
TOP_K = 5

os.makedirs(DATA_DIR, exist_ok=True)

## 3. Configuración de logs

In [11]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
for ruidoso in ("httpx", "urllib3", "sentence_transformers", "pinecone"):
    logging.getLogger(ruidoso).setLevel(logging.WARNING)
logger = logging.getLogger("rag")

## 4. Modelo de embeddings
La dimensión del índice se toma del modelo, para que siempre coincidan.

In [12]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDINGS_MODEL,
    encode_kwargs={"normalize_embeddings": True},
)

DIMENSION = len(embeddings.embed_query("prueba"))
logger.info("Embeddings: %s (dimensión %d)", EMBEDDINGS_MODEL, DIMENSION)

23:23:11 | INFO    | numexpr.utils | NumExpr defaulting to 2 threads.
23:23:23 | INFO    | datasets | TensorFlow version 2.21.0 available.
23:23:23 | INFO    | datasets | JAX version 0.11.1 available.


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

23:23:28 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

23:23:45 | INFO    | rag | Embeddings: intfloat/multilingual-e5-small (dimensión 384)


## 5. Setup de Pinecone: crear el índice si no existe

In [13]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])


def inicializar_indice():
    """Verifica si el índice existe; si no, lo crea en modo Serverless."""
    existentes = [i.name for i in pc.list_indexes()]

    if INDEX_NAME in existentes:
        logger.info("El índice '%s' ya existe", INDEX_NAME)
    else:
        logger.info("Creando índice serverless '%s' (dim=%d, %s/%s)...",
                    INDEX_NAME, DIMENSION, CLOUD, REGION)
        pc.create_index(
            name=INDEX_NAME,
            dimension=DIMENSION,
            metric="cosine",
            spec=ServerlessSpec(cloud=CLOUD, region=REGION),
        )
        logger.info("Índice creado")

    descripcion = pc.describe_index(INDEX_NAME)
    if descripcion.dimension != DIMENSION:
        raise ValueError(
            f"Mismatch de dimensiones: el índice tiene {descripcion.dimension} "
            f"y el modelo de embeddings genera {DIMENSION}"
        )
    logger.info("Dimensión verificada: %d", descripcion.dimension)
    return pc.Index(INDEX_NAME)


index = inicializar_indice()
print(index.describe_index_stats())

23:24:06 | INFO    | rag | Creando índice serverless 'curso-ai-engineering' (dim=384, aws/us-east-1)...
23:24:12 | INFO    | rag | Índice creado
23:24:12 | INFO    | rag | Dimensión verificada: 384


{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}


## 6. Dataset: documentación técnica de FastAPI
Ocho archivos Markdown. La primera línea de cada uno indica su categoría, que se guarda como metadato.

1

In [14]:
%%writefile data/instalacion.md
categoria: primeros-pasos
# Instalación y primer servidor

FastAPI se instala con pip. La forma recomendada es instalar el paquete con sus extras estándar: pip install "fastapi[standard]". Esto agrega Uvicorn, el servidor ASGI que ejecuta la aplicación, y la herramienta de línea de comandos de FastAPI.

Una aplicación mínima crea un objeto app = FastAPI() en el archivo main.py y define al menos una ruta. Para desarrollo se ejecuta con el comando fastapi dev main.py, que reinicia el servidor automáticamente cada vez que cambia el código. En producción se usa fastapi run main.py.

Por defecto el servidor escucha en el puerto 8000. Una de las ventajas de FastAPI es la documentación automática: al entrar a /docs se abre Swagger UI, una interfaz interactiva para probar cada endpoint, y en /redoc está la versión de ReDoc. Ambas se generan a partir del esquema OpenAPI que FastAPI construye solo, leyendo las rutas y los tipos de datos.

Writing data/instalacion.md


2

In [15]:
%%writefile data/rutas.md
categoria: rutas
# Rutas y operaciones

En FastAPI, cada endpoint se define con un decorador que indica el método HTTP y la ruta: @app.get, @app.post, @app.put, @app.patch y @app.delete. La función que está debajo del decorador se llama path operation function y su valor de retorno se convierte automáticamente a JSON.

Las funciones pueden declararse con def o con async def. Conviene usar async def cuando adentro se usan librerías asíncronas, por ejemplo un cliente HTTP o un driver de base de datos asíncrono, porque así el servidor puede atender otras peticiones mientras espera.

Cuando la aplicación crece, las rutas se organizan en módulos con APIRouter. Cada router agrupa endpoints relacionados y se incorpora a la aplicación principal con app.include_router. Al incluirlo se puede definir un prefix, como /usuarios, y una lista de tags, que agrupan los endpoints en la documentación de Swagger.

Writing data/rutas.md


3

In [16]:
%%writefile data/parametros.md
categoria: rutas
# Parámetros de ruta y de consulta

Los parámetros de ruta se escriben entre llaves en la URL, por ejemplo /items/{item_id}, y se reciben como argumentos de la función con el mismo nombre. Si se anota el tipo, como item_id: int, FastAPI convierte el valor automáticamente y rechaza la petición si no es un número.

Los argumentos de la función que no forman parte de la ruta se interpretan como parámetros de consulta (query parameters), los que van después del signo de pregunta en la URL, como ?skip=0&limit=10. Si tienen un valor por defecto son opcionales; si no, son obligatorios.

Para agregar validaciones se usan Query y Path. Por ejemplo, Query(max_length=50) limita el largo de un texto, y Path(ge=1) exige que un número sea mayor o igual a 1. Estas restricciones también aparecen en la documentación automática.

Writing data/parametros.md


4

In [17]:
%%writefile data/modelos_pydantic.md
categoria: validacion
# Cuerpo de la petición y modelos Pydantic

Para recibir datos en el cuerpo (body) de una petición POST o PUT se define una clase que hereda de BaseModel de Pydantic y se usa como tipo de un argumento de la función. FastAPI lee el JSON recibido, lo valida contra el modelo y entrega a la función un objeto ya validado.

Si el cuerpo no cumple con el modelo, por ejemplo porque falta un campo obligatorio o un valor tiene el tipo equivocado, FastAPI responde automáticamente con el código de estado 422 Unprocessable Entity y un detalle de los errores, sin ejecutar la función.

El parámetro response_model del decorador indica qué forma tiene la respuesta. FastAPI filtra la salida para devolver solo los campos de ese modelo, lo que sirve para no exponer datos sensibles como contraseñas. Con Field se agregan restricciones y descripciones a cada campo.

Writing data/modelos_pydantic.md


5

In [18]:
%%writefile data/dependencias.md
categoria: arquitectura
# Inyección de dependencias

FastAPI tiene un sistema de inyección de dependencias basado en Depends. Una dependencia es una función que FastAPI ejecuta antes del endpoint; su resultado se pasa como argumento. Se declara así: def listar(db = Depends(obtener_sesion)).

Las dependencias sirven para reutilizar lógica común: abrir una conexión a la base de datos, leer el usuario autenticado o validar parámetros de paginación. Una dependencia puede, a su vez, depender de otras, y FastAPI resuelve toda la cadena.

Si la dependencia usa yield en lugar de return, el código que está después del yield se ejecuta cuando termina la petición. Es la forma habitual de cerrar la sesión de la base de datos aunque haya ocurrido un error. También se pueden aplicar dependencias a un router completo con el parámetro dependencies, para que se ejecuten en todos sus endpoints.

Writing data/dependencias.md


6

In [19]:
%%writefile data/errores.md
categoria: errores
# Manejo de errores

Para devolver un error al cliente se lanza una HTTPException con un status_code y un detail. Por ejemplo, raise HTTPException(status_code=404, detail="Producto no encontrado") corta la ejecución del endpoint y responde con ese código y un JSON que contiene el mensaje personalizado.

El campo detail puede ser un texto, una lista o un diccionario. También se pueden agregar encabezados (headers) a la respuesta de error, algo útil en errores de autenticación.

Para cambiar cómo se responde ante cierto tipo de error se registra un manejador con el decorador @app.exception_handler. Así se puede capturar una excepción propia y convertirla en una respuesta con el formato que use la API. El mismo mecanismo permite personalizar la respuesta de RequestValidationError, que es la excepción que FastAPI lanza cuando los datos recibidos no pasan la validación.

Writing data/errores.md


7

In [20]:
%%writefile data/seguridad.md
categoria: seguridad
# Seguridad y autenticación

FastAPI incluye utilidades para implementar OAuth2. La clase OAuth2PasswordBearer define el esquema de autenticación con token: recibe el parámetro tokenUrl, que es la ruta donde el cliente envía usuario y contraseña para obtener el token. Usada como dependencia, extrae el token del encabezado Authorization: Bearer.

Si la petición no trae el token, FastAPI responde con el código 401 Unauthorized. El token suele ser un JWT firmado que contiene el identificador del usuario y una fecha de vencimiento.

Las contraseñas nunca se guardan en texto plano: se almacena un hash generado con un algoritmo seguro como bcrypt o Argon2. Al iniciar sesión se compara el hash de la contraseña ingresada con el guardado. Una dependencia como obtener_usuario_actual combina la lectura del token con la búsqueda del usuario y se reutiliza en todos los endpoints protegidos.

Writing data/seguridad.md


8

In [21]:
%%writefile data/middleware_cors.md
categoria: middleware
# Middleware y CORS

Un middleware es una función que se ejecuta en cada petición antes de llegar al endpoint y en cada respuesta antes de volver al cliente. Se define con el decorador @app.middleware("http") y sirve, por ejemplo, para medir el tiempo de respuesta o agregar encabezados.

CORS (Cross-Origin Resource Sharing) es el mecanismo que permite que un frontend alojado en otro origen, como una aplicación React en localhost:3000, llame a la API. Por seguridad, el navegador bloquea esas llamadas salvo que el servidor las autorice.

En FastAPI se habilita con CORSMiddleware: app.add_middleware(CORSMiddleware, allow_origins=["http://localhost:3000"], allow_methods=["*"], allow_headers=["*"]). En allow_origins conviene listar solo los orígenes permitidos en lugar de usar un asterisco, sobre todo si allow_credentials está activado.

Writing data/middleware_cors.md


## 7. Carga de documentos y chunking
Cada chunk lleva metadatos: fuente, título, categoría y página (número de fragmento dentro del archivo).

In [22]:
import tiktoken
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
encoder = tiktoken.get_encoding("cl100k_base")


def cargar_documentos(carpeta: str = DATA_DIR) -> list[Document]:
    """Lee los .md de la carpeta. La primera línea indica la categoría."""
    documentos = []
    for archivo in sorted(os.listdir(carpeta)):
        if not archivo.endswith(".md"):
            continue
        with open(os.path.join(carpeta, archivo), encoding="utf-8") as f:
            lineas = f.read().strip().splitlines()

        categoria = lineas[0].split(":", 1)[1].strip() if lineas[0].startswith("categoria:") else "general"
        contenido = "\n".join(lineas[1:]).strip()
        titulo = next((l.lstrip("# ").strip() for l in lineas if l.startswith("# ")), archivo)

        documentos.append(Document(
            page_content=contenido,
            metadata={"fuente": archivo, "titulo": titulo, "categoria": categoria},
        ))
    logger.info("Documentos cargados: %d", len(documentos))
    return documentos


def fragmentar(documentos: list[Document]) -> list[Document]:
    """Divide cada documento y numera sus fragmentos como páginas."""
    chunks = []
    for doc in documentos:
        for pagina, texto in enumerate(splitter.split_text(doc.page_content), start=1):
            metadata = {**doc.metadata, "pagina": pagina,
                        "chunk_id": f"{doc.metadata['fuente']}#{pagina}"}
            chunks.append(Document(page_content=texto, metadata=metadata))
    logger.info("Chunks generados: %d", len(chunks))
    return chunks


chunks = fragmentar(cargar_documentos())
for c in chunks:
    print(f"{c.metadata['chunk_id']:<25} {c.metadata['categoria']:<15} "
          f"{len(encoder.encode(c.page_content))} tokens")

23:28:07 | INFO    | rag | Documentos cargados: 8
23:28:07 | INFO    | rag | Chunks generados: 8


dependencias.md#1         arquitectura    204 tokens
errores.md#1              errores         198 tokens
instalacion.md#1          primeros-pasos  229 tokens
middleware_cors.md#1      middleware      199 tokens
modelos_pydantic.md#1     validacion      199 tokens
parametros.md#1           rutas           214 tokens
rutas.md#1                rutas           214 tokens
seguridad.md#1            seguridad       210 tokens


## 8. Ingesta en Pinecone (con namespace)
El texto original se guarda en la metadata de cada vector (campo "texto"). Si el namespace ya tiene datos, no se vuelve a subir.

In [23]:
import time
from langchain_pinecone import PineconeVectorStore


def contar_vectores() -> int:
    stats = index.describe_index_stats()
    ns = (stats.namespaces or {}).get(NAMESPACE)
    return ns.vector_count if ns else 0


def crear_vectorstore() -> PineconeVectorStore:
    return PineconeVectorStore(
        index=index,
        embedding=embeddings,
        namespace=NAMESPACE,
        text_key="texto",   # nombre del campo de metadata donde va el texto original
    )


def ingestar(forzar: bool = False) -> PineconeVectorStore:
    vectorstore = crear_vectorstore()
    existentes = contar_vectores()

    if existentes > 0 and not forzar:
        logger.info("El namespace '%s' ya tiene %d vectores -> se omite la ingesta",
                    NAMESPACE, existentes)
        return vectorstore

    if forzar and existentes > 0:
        logger.info("Reindexando: se borran los vectores del namespace '%s'", NAMESPACE)
        index.delete(delete_all=True, namespace=NAMESPACE)

    chunks = fragmentar(cargar_documentos())
    ids = [c.metadata["chunk_id"] for c in chunks]
    vectorstore.add_documents(chunks, ids=ids)
    logger.info("Subidos %d chunks a Pinecone (namespace '%s')", len(chunks), NAMESPACE)

    # Pinecone tarda unos segundos en reflejar los vectores nuevos
    for _ in range(30):
        if contar_vectores() >= len(chunks):
            break
        time.sleep(2)
    logger.info("Vectores disponibles en el namespace: %d", contar_vectores())
    return vectorstore


vectorstore = ingestar()

23:28:30 | INFO    | rag | Documentos cargados: 8
23:28:30 | INFO    | rag | Chunks generados: 8
23:28:33 | INFO    | rag | Subidos 8 chunks a Pinecone (namespace 'fastapi-docs')
23:28:33 | INFO    | rag | Vectores disponibles en el namespace: 8


## 9. Prueba de persistencia

In [24]:
vectorstore = ingestar()

23:28:51 | INFO    | rag | El namespace 'fastapi-docs' ya tiene 8 vectores -> se omite la ingesta


## 10. Verificación de la metadata de un vector

In [25]:
resultado = index.fetch(ids=["errores.md#1"], namespace=NAMESPACE)
metadata = resultado.vectors["errores.md#1"].metadata

for clave, valor in metadata.items():
    texto = str(valor)
    print(f"{clave:<10}: {texto[:80]}{'...' if len(texto) > 80 else ''}")

categoria : errores
chunk_id  : errores.md#1
fuente    : errores.md
pagina    : 1.0
texto     : # Manejo de errores

Para devolver un error al cliente se lanza una HTTPExceptio...
titulo    : Manejo de errores


## 11. Clase RAGSystem: recuperador híbrido (BM25 + vectorial)
Combina la búsqueda léxica (BM25, buena para términos exactos como HTTPException) con la semántica de Pinecone usando un EnsembleRetriever.

In [26]:
import re
from langchain_community.retrievers import BM25Retriever

try:
    from langchain_classic.retrievers import EnsembleRetriever
except ImportError:
    from langchain.retrievers import EnsembleRetriever


def tokenizar(texto: str) -> list[str]:
    """Minúsculas y sin signos, para que 'HTTPException.' coincida con 'httpexception'."""
    return re.findall(r"\w+", texto.lower())


class RAGSystem:
    """Encapsula un EnsembleRetriever que combina BM25 (léxico) y Pinecone (semántico)."""

    def __init__(self, vectorstore, documentos: list[Document], k: int = TOP_K,
                 pesos: tuple[float, float] = (0.5, 0.5)):
        self.k = k
        self.vector_retriever = vectorstore.as_retriever(search_kwargs={"k": k})

        self.bm25_retriever = BM25Retriever.from_documents(documentos, preprocess_func=tokenizar)
        self.bm25_retriever.k = k

        self.hybrid_retriever = EnsembleRetriever(
            retrievers=[self.bm25_retriever, self.vector_retriever],
            weights=list(pesos),
        )
        self.retrievers = {
            "bm25": self.bm25_retriever,
            "vectorial": self.vector_retriever,
            "hibrido": self.hybrid_retriever,
        }
        logger.info("RAGSystem listo: k=%d, pesos BM25/vectorial=%s, %d documentos para BM25",
                    k, pesos, len(documentos))

    async def buscar(self, query: str, modo: str = "hibrido") -> list[Document]:
        """Devuelve los top-k documentos para la consulta."""
        if modo not in self.retrievers:
            raise ValueError(f"Modo inválido: {modo!r}. Opciones: {list(self.retrievers)}")
        docs = (await self.retrievers[modo].ainvoke(query))[: self.k]
        logger.info("[%s] '%s' -> %s", modo, query, [d.metadata["fuente"] for d in docs])
        return docs


rag = RAGSystem(vectorstore, documentos=fragmentar(cargar_documentos()))

/tmp/ipykernel_3186/1872930707.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever
23:32:16 | INFO    | rag | Documentos cargados: 8
23:32:16 | INFO    | rag | Chunks generados: 8
23:32:16 | INFO    | rag | RAGSystem listo: k=5, pesos BM25/vectorial=(0.5, 0.5), 8 documentos para BM25


## 12. Prueba del recuperador: comparación de los tres modos

In [27]:
consulta = "¿Cómo devuelvo un error 404 con un mensaje personalizado?"

for modo in ("bm25", "vectorial", "hibrido"):
    docs = await rag.buscar(consulta, modo=modo)
    print(f"\n=== {modo.upper()} ===")
    for posicion, d in enumerate(docs, start=1):
        print(f"{posicion}. {d.metadata['fuente']:<22} [{d.metadata['categoria']}]")

23:32:49 | INFO    | rag | [bm25] '¿Cómo devuelvo un error 404 con un mensaje personalizado?' -> ['errores.md', 'dependencias.md', 'rutas.md', 'modelos_pydantic.md', 'seguridad.md']



=== BM25 ===
1. errores.md             [errores]
2. dependencias.md        [arquitectura]
3. rutas.md               [rutas]
4. modelos_pydantic.md    [validacion]
5. seguridad.md           [seguridad]


23:32:49 | INFO    | rag | [vectorial] '¿Cómo devuelvo un error 404 con un mensaje personalizado?' -> ['errores.md', 'modelos_pydantic.md', 'parametros.md', 'middleware_cors.md', 'seguridad.md']



=== VECTORIAL ===
1. errores.md             [errores]
2. modelos_pydantic.md    [validacion]
3. parametros.md          [rutas]
4. middleware_cors.md     [middleware]
5. seguridad.md           [seguridad]


23:32:50 | INFO    | rag | [hibrido] '¿Cómo devuelvo un error 404 con un mensaje personalizado?' -> ['errores.md', 'modelos_pydantic.md', 'seguridad.md', 'dependencias.md', 'rutas.md']



=== HIBRIDO ===
1. errores.md             [errores]
2. modelos_pydantic.md    [validacion]
3. seguridad.md           [seguridad]
4. dependencias.md        [arquitectura]
5. rutas.md               [rutas]


## 13. Golden Set: 5 preguntas con su documento fuente conocido

In [28]:
%%writefile golden_set.json
[
  {"pregunta": "¿Qué hace OAuth2PasswordBearer y para qué sirve tokenUrl?",
   "documento_id_esperado": "seguridad.md"},
  {"pregunta": "Mi frontend en React no puede llamar a la API porque el navegador bloquea la petición",
   "documento_id_esperado": "middleware_cors.md"},
  {"pregunta": "¿Qué código de estado responde FastAPI cuando falta un campo obligatorio en el JSON?",
   "documento_id_esperado": "modelos_pydantic.md"},
  {"pregunta": "¿Cómo cierro la sesión de la base de datos al terminar cada petición?",
   "documento_id_esperado": "dependencias.md"},
  {"pregunta": "¿Dónde veo la documentación interactiva de mis endpoints?",
   "documento_id_esperado": "instalacion.md"}
]

Writing golden_set.json


## 14. Evaluación: Precision@5 y Recall@5
- Recall@5: 1 si el documento esperado está entre los 5 recuperados, 0 si no.
- Precision@5: proporción de los 5 recuperados que pertenecen al documento esperado.
- Posición / MRR: en qué lugar aparece el documento correcto (1 = primero).

In [29]:
import json
from collections import Counter

with open("golden_set.json", encoding="utf-8") as f:
    golden_set = json.load(f)

chunks_por_documento = Counter(c.metadata["fuente"] for c in fragmentar(cargar_documentos()))


async def evaluar(modo: str, k: int = TOP_K) -> dict:
    """Corre el Golden Set con un modo de búsqueda y calcula las métricas."""
    filas = []
    for item in golden_set:
        esperado = item["documento_id_esperado"]
        docs = await rag.buscar(item["pregunta"], modo=modo)
        fuentes = [d.metadata["fuente"] for d in docs]

        relevantes = sum(f == esperado for f in fuentes)
        posicion = fuentes.index(esperado) + 1 if esperado in fuentes else None
        filas.append({
            "pregunta": item["pregunta"],
            "esperado": esperado,
            "posicion": posicion,
            "recall": 1.0 if relevantes > 0 else 0.0,
            "precision": relevantes / k,
            "precision_max": min(chunks_por_documento[esperado], k) / k,
            "rr": 1 / posicion if posicion else 0.0,
        })

    n = len(filas)
    return {
        "modo": modo,
        "filas": filas,
        "recall": sum(f["recall"] for f in filas) / n,
        "precision": sum(f["precision"] for f in filas) / n,
        "precision_max": sum(f["precision_max"] for f in filas) / n,
        "mrr": sum(f["rr"] for f in filas) / n,
    }


resultados = {modo: await evaluar(modo) for modo in ("bm25", "vectorial", "hibrido")}

23:34:03 | INFO    | rag | Documentos cargados: 8
23:34:03 | INFO    | rag | Chunks generados: 8
23:34:03 | INFO    | rag | [bm25] '¿Qué hace OAuth2PasswordBearer y para qué sirve tokenUrl?' -> ['modelos_pydantic.md', 'seguridad.md', 'middleware_cors.md', 'instalacion.md', 'errores.md']
23:34:03 | INFO    | rag | [bm25] 'Mi frontend en React no puede llamar a la API porque el navegador bloquea la petición' -> ['middleware_cors.md', 'errores.md', 'rutas.md', 'modelos_pydantic.md', 'dependencias.md']
23:34:03 | INFO    | rag | [bm25] '¿Qué código de estado responde FastAPI cuando falta un campo obligatorio en el JSON?' -> ['modelos_pydantic.md', 'errores.md', 'dependencias.md', 'rutas.md', 'seguridad.md']
23:34:03 | INFO    | rag | [bm25] '¿Cómo cierro la sesión de la base de datos al terminar cada petición?' -> ['dependencias.md', 'errores.md', 'rutas.md', 'seguridad.md', 'modelos_pydantic.md']
23:34:03 | INFO    | rag | [bm25] '¿Dónde veo la documentación interactiva de mis endpoints?'

## 15. Reporte de resultados

In [30]:
print("=" * 78)
print("DETALLE POR PREGUNTA (posición del documento correcto en el top-5; '-' = no apareció)")
print("=" * 78)
print(f"{'Esperado':<22}{'BM25':>8}{'Vectorial':>12}{'Híbrido':>10}   Pregunta")
for i, item in enumerate(golden_set):
    pos = [resultados[m]["filas"][i]["posicion"] or "-" for m in ("bm25", "vectorial", "hibrido")]
    print(f"{item['documento_id_esperado']:<22}{pos[0]!s:>8}{pos[1]!s:>12}{pos[2]!s:>10}   "
          f"{item['pregunta'][:45]}...")

print("\n" + "=" * 78)
print("RESUMEN")
print("=" * 78)
print(f"{'Modo':<12}{'Recall@5':>10}{'Precision@5':>14}{'Precision máx.':>17}{'MRR':>8}")
for modo, r in resultados.items():
    print(f"{modo:<12}{r['recall']:>10.2f}{r['precision']:>14.2f}{r['precision_max']:>17.2f}{r['mrr']:>8.2f}")

print(f"\nDocumentos indexados: {len(chunks_por_documento)} | Chunks: {sum(chunks_por_documento.values())} | k = {TOP_K}")
print("Precision máx.: el mejor valor posible, porque cada documento tiene un solo chunk "
      "y en el top-5 solo puede haber 1 relevante.")

DETALLE POR PREGUNTA (posición del documento correcto en el top-5; '-' = no apareció)
Esperado                  BM25   Vectorial   Híbrido   Pregunta
seguridad.md                 2           1         1   ¿Qué hace OAuth2PasswordBearer y para qué sir...
middleware_cors.md           1           1         1   Mi frontend en React no puede llamar a la API...
modelos_pydantic.md          1           1         1   ¿Qué código de estado responde FastAPI cuando...
dependencias.md              1           1         1   ¿Cómo cierro la sesión de la base de datos al...
instalacion.md               1           2         1   ¿Dónde veo la documentación interactiva de mi...

RESUMEN
Modo          Recall@5   Precision@5   Precision máx.     MRR
bm25              1.00          0.20             0.20    0.90
vectorial         1.00          0.20             0.20    0.90
hibrido           1.00          0.20             0.20    1.00

Documentos indexados: 8 | Chunks: 8 | k = 5
Precision máx.: el mejor val

In [31]:
!zip -r datos_pre4.zip data golden_set.json

from google.colab import files
files.download("datos_pre4.zip")

  adding: data/ (stored 0%)
  adding: data/modelos_pydantic.md (deflated 44%)
  adding: data/errores.md (deflated 44%)
  adding: data/parametros.md (deflated 43%)
  adding: data/dependencias.md (deflated 46%)
  adding: data/instalacion.md (deflated 43%)
  adding: data/middleware_cors.md (deflated 42%)
  adding: data/seguridad.md (deflated 44%)
  adding: data/rutas.md (deflated 44%)
  adding: golden_set.json (deflated 50%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>